# RNV U-Net training on Colab

The adopted recipe (40 epochs, cosine learning rate, last epoch, no inner validation) is `docs/results/2026-11-07-train-budget.md`; it comes from `cfg.Net` through the bundle. Put on Google Drive, under `MyDrive/rnv/`:

- `colab_bundle/`, the output of `run_export` (`cnn/result/DRACTUH_unclustered/colab_bundle/`). Re-export after any `cfg.Net` change: an old bundle trains the old recipe, or refuses.
- `colab/`, holding `rnvtorch.py` and `test_parity.py` from the repo

Runtime → Change runtime type → GPU. Train **one fold per session**. Every epoch is checkpointed to Drive, so after a disconnect, rerun the same cell and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rnv'
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Local disk reads faster than Drive; the bundle is ~620 MB.
!rsync -a {DRIVE}/colab_bundle/ /content/bundle/ && du -sh /content/bundle

**Stages 1–2 (optional, minutes).** Checks that the code and bundle on Drive are the ones that passed locally.

In [ ]:
!python {DRIVE}/colab/test_parity.py /content/bundle

**Stage 3: one-fold smoke run.** Trains fold 1 for 1 epoch, stops, resumes to epoch 2, then writes `P`. It checks the cosine schedule, the last-epoch output and the patch count, and prints seconds per epoch and a projected time for one CV.

In [ ]:
!rm -rf {DRIVE}/runs/smoke
!python {DRIVE}/colab/test_parity.py stage3 /content/bundle {DRIVE}/runs/smoke 2

**CV training.** One run of folds 1–5, about 3.4 h per fold on an L4. `RNG` seeds the random draws (patch list, initial weights, jitter, flips, shuffle, dropout); it is not a localization seed. After a disconnect, rerun the cell: it resumes the unfinished fold. A checkpoint trained with other settings refuses to resume.

**Stall rule** (`docs/briefs/train-budget.md`): before scoring, read every `fold*/log.csv`. A fold whose mean training loss over its last five epochs exceeds 0.35 is retrained once from scratch with the next `RNG`, into its own run root, and that retrain is used whatever it scores.

Download `runs/cv_rng<R>/` (`probcache/` and every `fold*/log.csv`) and score it locally:

```matlab
run_cache(cnngvf.defaults(), "<downloaded probcache>")
```

In [ ]:
RNG = 0
for FOLD in range(1, 6):
    !python {DRIVE}/colab/rnvtorch.py train /content/bundle {DRIVE}/runs/cv_rng{RNG} --fold {FOLD} --rng {RNG} --workers 2

**Experiment: arm AB, the manuscript's recipe** (`docs/briefs/paper-adoption.md`, branch `exp/paper-recipe`). `--recipe paper` trains a pretrained EfficientNet-B3 U-Net with CE + Dice, minibatch 8, bf16, rate 1e-4 → 1e-6, and affine + CLAHE augmentation; everything else is the adopted recipe. Run the smoke first: it checks the warp, the resume and the cosine, and prints the projected CV time.

**Stall rule for this arm:** a fold whose last-five-epoch mean training loss exceeds 1.25 × the median of the five folds' is retrained once with the next `RNG`, and that retrain is used whatever it scores.

In [ ]:
!pip install -q segmentation-models-pytorch==0.5.0
!rm -rf {DRIVE}/runs/smoke_paper
!python {DRIVE}/colab/test_parity.py stage3 /content/bundle {DRIVE}/runs/smoke_paper 2 paper

In [ ]:
RNG = 0
for FOLD in range(1, 6):
    !python {DRIVE}/colab/rnvtorch.py train /content/bundle {DRIVE}/runs/paper_rng{RNG} --fold {FOLD} --rng {RNG} --workers 2 --recipe paper